# Train a Driving Agent on Oasis 3 (Colab)

Oasis 3 is a hosted, learned driving simulator: send it actions (throttle + steering) and it
generates the next camera frames — so it's a world you can do reinforcement learning in. This
notebook wraps it as a [Gymnasium](https://gymnasium.farama.org/) env and trains a small
[Stable-Baselines3](https://stable-baselines3.readthedocs.io/) PPO policy to drive in it.

Only the **PPO policy** is trained; Oasis (the simulator) and the depth model (the reward) are
frozen and hosted.

**To run:** Runtime → Change runtime type → **GPU**, set your `DECART_API_KEY` (see §1), then
**Run all**.

## 1. Setup

This installs **`decart-oasis`** — the **SDK** (the Oasis connection, `A2VClient`) — from PyPI, then
clones and installs **`oasis_demo`** — **everything you can play with**: the depth/collision reward,
the Gymnasium env, the policy + training, and the live preview. `oasis_demo` isn't on PyPI, so it's
installed **editable** from the clone — open `extras/oasis_demo/` in the file browser, tweak it (e.g.
`rl/reward.py`), restart the runtime, and re-run.

Add your **`DECART_API_KEY`** in the Colab secrets panel (🔑, left sidebar) — it's required to
connect. Then **Run all**.

In [ ]:
import importlib, os, subprocess, sys

# The SDK (`decart-oasis`) is installed from the public PyPI package. `oasis_demo` — the extras you
# can play with (reward, env, preview) — isn't published, so we clone it from GitHub and install it
# editable, so you can tweak e.g. `extras/oasis_demo/rl/reward.py`, restart the runtime, and re-run.
REPO_URL = "https://github.com/DecartAI/decart-robotics.git"
REPO_DIR = REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")

if importlib.util.find_spec("oasis_demo") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "decart-oasis"], check=True)
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_DIR}/extras"],
                   check=True)
    # Editable installs register via a .pth that's only read at interpreter startup, so put the
    # source on sys.path to import it now without restarting the runtime.
    sys.path.insert(0, os.path.abspath(f"{REPO_DIR}/extras"))
importlib.invalidate_caches()

# API key (required to connect). Set it in the Colab secrets panel (🔑) as DECART_API_KEY,
# or here with os.environ["DECART_API_KEY"] = "...".
if not os.environ.get("DECART_API_KEY"):
    try:
        from google.colab import userdata
        os.environ["DECART_API_KEY"] = userdata.get("DECART_API_KEY") or ""
    except Exception:
        pass
print("DECART_API_KEY set:", bool(os.environ.get("DECART_API_KEY")))

from decart_oasis import A2VClient
from oasis_demo.live_preview import LiveCameraPreview
print("Ready.")

### Compute device

GPU → reward/policy run on CUDA; CPU-only still works, just slower (Oasis API latency dominates).

In [ ]:
import torch

DEVICE = 0 if torch.cuda.is_available() else -1
print(f"CUDA available: {torch.cuda.is_available()}  ->  reward/depth device = {DEVICE}")
if DEVICE == -1:
    print("Tip: in Colab, Runtime > Change runtime type > GPU for a faster depth model.")

## 2. Drive the simulator by hand

A quick check that the API works: we send a fixed chunk of four `[throttle, steering]` actions and
stream the returned frames into a live `left | front | right` video, with a playback-synced
collision-risk bar beneath it. The first chunk loads the depth model, so it pauses briefly.

In [ ]:
import numpy as np

from decart_oasis import A2VClient
from oasis_demo.live_preview import LiveCameraPreview
from oasis_demo.rl.reward import DepthCollisionReward

# `prompt` sets the scene Oasis generates; try changing it (e.g. "driving on a highway").
prompt = "driving in an urban area"

# One chunk = four [throttle, steering] actions, each in [-1, 1] (throttle forward,
# steering negative=left / positive=right). Here we hand-pick a gentle forward-with-slight-
# weave chunk just to exercise the API; during training the policy chooses these instead.
action_chunk = np.array(
    [[0.15, -0.10], [0.20, -0.04], [0.22, 0.04], [0.18, 0.10]],
    dtype=np.float32,
)

# The depth-based reward, created here so the preview can score collision risk per frame;
# reused in the reward section below. DEVICE was auto-selected above (0=CUDA, -1=CPU).
# `straightness_weight` is the knob to sweep for the speed-vs-swerve sweet spot: start at 0.0
# (no turn penalty) to confirm it accelerates, then try 0.05 / 0.1 / 0.2. Widen the turn
# allowance with `heading_free_play=...` if it's too eager to punish turning.
reward_fn = DepthCollisionReward(device=DEVICE, straightness_weight=0.1)

# `frame_consumer=preview` streams every generated chunk into the live video; `risk_fn=...`
# adds the playback-synced risk bar. The depth model loads lazily on the first scored frame.
with LiveCameraPreview(risk_fn=reward_fn.collision_risk) as preview, A2VClient(
    timeout=120.0, frame_consumer=preview
) as client:
    client.prompt(prompt)  # set the scene; resets the world model's rollout
    for _ in range(12):
        result = client.infer(action_chunk)  # send actions, get the next frames back


## 3. The reward

RL needs a reward — a score per step the agent learns to maximize. Ours rewards **forward progress**,
penalizes **net turning** (so it weaves around obstacles instead of circling), and **terminates** the
episode on a likely collision (a depth model on the `front` frame measures how much of the scene is
dangerously close). The cell runs it on the §2 frames and shows the depth map, `reward`,
`terminated`, and `collision_risk`. Tune the constants in `extras/oasis_demo/rl/reward.py`.

In [ ]:
# reward_fn (DepthCollisionReward) was created in the Quick API check cell above.
front_frames = result.frames["front"]
reward_fn.display_depth(front_frames)  # shows depth + the ignored ego-region boundary

reward, terminated, info = reward_fn(front_frames, action_chunk)
print(f"reward={reward:.4f}  terminated={terminated}  collision_risk={info['collision_risk']:.4f}")

## 4. (Optional) Warm-start with behavior cloning

Clone the policy on recordings of *people* driving Oasis, so it starts from human-like driving
instead of random exploration. This downloads our demo episodes (**~2.2 GB**, a few minutes; cached
afterwards) and behavior-clones the policy; §5 then continues it with RL automatically.

In [ ]:
from oasis_demo.rl.sft import pretrain_ppo_from_demo

# Downloads our demo episodes and behavior-clones the PPO policy §5 trains (same network + config),
# leaving it in `model`. device="auto" uses the GPU when present.
model = pretrain_ppo_from_demo(device="auto", epochs=5, batch_size=16, learning_rate=3e-4)
print("Warm-start done — §5 will continue this policy with RL.")

## 5. Train the policy with RL

PPO drives the agent in Oasis, scores each step with the §3 reward, and improves the policy —
continuing the §4 warm-started policy if it exists, else training from scratch. Every step is a live
Oasis call, so `TOTAL_TIMESTEPS` is small (`500`): enough to see the loop, not to drive well. The
preview shows the latest episode as it trains.

In [ ]:
from stable_baselines3 import PPO

from oasis_demo.rl.env import OasisA2VEnv
from oasis_demo.rl.sb3 import torch_device_status
from oasis_demo.rl.sft import POLICY_KWARGS, PPO_HYPERPARAMS

PROMPT = "driving in an urban area"
TOTAL_TIMESTEPS = 500  # demo-sized; raise for a real run

with LiveCameraPreview() as preview, OasisA2VEnv(
    client_factory=lambda: A2VClient(frame_consumer=preview),
    reward_fn=reward_fn,
    prompt=PROMPT,
    max_steps=256,
) as env:
    # Continue the §4 warm-started policy if it exists, else train a fresh one (same PPO config).
    if "model" in globals() and isinstance(model, PPO):
        model.set_env(env)
        print("Continuing from the warm-started policy.")
    else:
        model = PPO("MultiInputPolicy", env, policy_kwargs=POLICY_KWARGS, **PPO_HYPERPARAMS)
        print("Training a fresh policy from scratch.")
    print(f"PPO policy device: {model.policy.device}")
    print(f"Feature extractor devices: {torch_device_status(model.policy.features_extractor)}")

    model.learn(total_timesteps=TOTAL_TIMESTEPS)
    print("Training done")
    model.save("oasis3_preview_ppo")

## 6. Watch it drive

One clean, deterministic episode of the policy you trained in §5. Set `USE_OUR_POLICY = True` in
the cell to watch our simple pretrained policy instead. 

In [ ]:
from oasis_demo.rl.env import OasisA2VEnv
from oasis_demo.rl.pretrained import load_pretrained_policy

USE_OUR_POLICY = False  # set this to True to watch our simple pretrained policy instead 

PROMPT = "driving in an urban area"

if USE_OUR_POLICY:
    model = load_pretrained_policy(device="auto")  # downloads our .pt once, then cached
# else: reuse `model` from §5

# Separate depth detector for the preview's risk bar so it doesn't share the env's reward model
# across threads. (§2 must have run: it imports A2VClient / LiveCameraPreview / DepthCollisionReward.)
risk_detector = DepthCollisionReward(device=DEVICE)

with LiveCameraPreview(risk_fn=risk_detector.collision_risk) as preview, OasisA2VEnv(
    client_factory=lambda: A2VClient(frame_consumer=preview),
    reward_fn=reward_fn,
    prompt=PROMPT,
    max_steps=256,
) as env:
    obs, _ = env.reset()  # reset() prompts with PROMPT; don't prompt again or the scene switches
    done = False
    total = 0.0
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        total += reward
        done = terminated or truncated
    print(
        f"episode reward={total:.3f}  collision_risk={info['collision_risk']:.3f}  "
        f"heading={info['heading']:.2f}  straightness_penalty={info['straightness_penalty']:.3f}"
    )

## What to try next

- **Train longer** — raise `TOTAL_TIMESTEPS` (wall-clock and quota scale with it).
- **Change the scene** — edit `PROMPT` (e.g. `"driving on a highway at sunset"`).
- **Reshape the reward** — edit the constants in `extras/oasis_demo/rl/reward.py`.
- **Drive it yourself** — change `action_chunk` in §2.
- **Hack the extras** — `oasis_demo` is installed editable; edit anything under `extras/oasis_demo/` (reward, env, policy), restart the runtime, and re-run.